# Limpieza y estandarización de catalogos

In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
from itertools import product
from numpy import nan
from helper_pandas import df_to_dict, simplify_string

Cargar Bitacoras onomásticas y funciones auxiliares

In [2]:
# Break and create dataframe
def break_and_simplify(string, sort=False):
    string = string.replace(' x ','+').replace(' y ','+')
    string = string.split('+')
    string = [simplify_string(s, str_case=None) for s in string if len(s)>0]
    if sort: string = sorted(string, sort=True)
    return string

def is_missing(s, dic):
    return ~s.isin(dic.values()) & s.notna()

def highlight_missing(s, dic, color='yellow'):
    miss = is_missing(s,dic)
    return ['background-color: yellow' if v else '' for v in miss]

def join_series(series, join_str='+', sort=False):
    series = series.dropna()
    if sort:  series = series.drop_duplicates().sort_values()
    series = join_str.join(series.to_list())
    if len(series)==0: series = nan
    return series

def first_n_to_text(s, n=5, join_str=','):
    d = s.head(n).to_list()
    d = join_str.join(d)
    return d

In [3]:
from joblib import load


from joblib import load

in_file = "data_clean/HBO_clean.pkl"
with open(in_file, 'rb') as f:
     df = load(f)

In [4]:
from joblib import load

in_file = 'data_clean/HBO_clean.pkl'
with open(in_file, 'rb') as f:
     df = load(f)  
df = df.reset_index()
df.tail()

,ID,Numero_progresivo_transcrito,Nombre_completo_transcrito,Primer_apellido,Segundo_apellido,Nombres_propios,Fecha_transcrito,Fecha_estandar,Fecha_año,Fecha_mes,...,Diagnostico_extendido,Procedencia_transcrito,Procedencia_estandar,Procedencia_extendido,Procedencia_alcaldia,Numero_acta_transcrito,Expediente_SEMEFO_transcrito,Bitacora_ingresos,Foja_transcrito,Observaciones
96839,BO_1982_07489,S-D,placenta,NaN,NaN,NaN,1982-06-05,1982-06-05,1982.0,6.0,...,NaN,15a,GAM-2,Coord Territorial 2 Gustavo A. Madero (Col Ara...,Gustavo A. Madero,960,3079,semefo_df_bo_1982,156,NaN
96840,BO_1982_07490,S-D,5 dedos del pie derecho de desconocido,NaN,NaN,NaN,1982-06-05,1982-06-05,1982.0,6.0,...,NaN,32a,COY-2,Coord Territorial 2 Coyoacan (Col Romero de Te...,Coyoacan,950,3060,semefo_df_bo_1982,156,NaN
96841,BO_1982_07491,S-D,dedo de desconocido,NaN,NaN,NaN,1982-11-19,1982-11-19,1982.0,11.0,...,NaN,32a,COY-2,Coord Territorial 2 Coyoacan (Col Romero de Te...,Coyoacan,2005,6389,semefo_df_bo_1982,156,NaN
96842,BO_1982_07492,S-D,4 dedos de desconocido,NaN,NaN,NaN,1982-11-28,1982-11-28,1982.0,11.0,...,NaN,27a,XO-2,Coord Territorial 2 Xochimilco (Barrio de San ...,Xochimilco,959,6528,semefo_df_bo_1982,156,NaN
96843,BO_1982_07493,S-D,osamenta de desconocido,NaN,NaN,NaN,1982-10-11,1982-10-11,1982.0,10.0,...,NaN,9a,MH-1,Coord Territorial 1 Miguel Hidalgo – Hospital ...,Miguel Hidalgo,4193,5629,semefo_df_bo_1982,156,no se recibio necropsia. es una osamenta.


## Revisar diagnosticos

**Metodología**
* Los diagnosticos se dividieron usando '+'
* Se determinaron las partes de diagnosticos y se resolvieron por frecuencia
    - AA es anemia o asfixia 
* Se determinaron diagnosticos equivalentes
    - AOV y AOVA
* Se genero diagnosticos simplificados estandar y clasifico
* Se reviso los diagnosticos no solubles y se propuso separaciones alternativas, por patron de espacio, x y similitud de string
    - AnxEmbarazo -> An+Embarazo
* Proceso iterativo con los faltantes

**Faltantes**
* P, E, CS, CQ, A, L, PTE, T, IP, M, H, I, EH, IA, MP

In [5]:
df['Diagnostico_transcrito'].nunique()

3709

### cat_diag_extendido.csv
* Diagnostico | Extendido | Clasificacion | Frecuencia

In [6]:
df_ext = pd.read_csv('catalogos/cat_diag_extendido.csv')
df_ext

display(df_ext.head())

# Revisar llaves unicas
df_ext['Diagnostico'].nunique()
print("Llaves unicas {}/{}".format( df_ext['Diagnostico'].nunique(), df_ext.shape[0]))

# Revisar llaves unicas
df_ext['Diagnostico'].nunique()
print("Extend unicas {}/{}".format( df_ext['Extendido'].nunique(), df_ext.shape[0]))

# Faltantes
print('Faltantes:', df_ext.isna().sum())

dic_ext = df_to_dict(df_ext[['Diagnostico','Extendido']])
len( dic_ext )

dic_cla = df_to_dict(df_ext[['Diagnostico','Clasificacion']])
len( dic_cla )

,Diagnostico,Extendido,Clasificacion
0,AQ,amputacion quirurgica,Amputación
1,AQXT,amputacion quirurgica por traumatismo,Amputación
2,AT,amputacion traumatica,Amputación
3,AQXTMi,amputacion quirurgica por traumatismo miembro,Amputación
4,AQxTrat,amputacion quirurgica por tratamiento,Amputación


Llaves unicas 244/244
Extend unicas 244/244
Faltantes: Diagnostico       0
Extendido         0
Clasificacion    79
dtype: int64


244

### cat_diag_sustitucion.xlsx
* Diagnostico_transcrito | Diagnostico_estandar | Freq | Npartes | Nfalta | Partes coloreadas | IDs

In [7]:
df_sus = pd.read_excel('catalogos/cat_diag_sustitucion.xlsx')
# quitar remplazos iguales
df_sus = df_sus[df_sus['Diagnostico_transcrito']!=df_sus['Diagnostico_estandar']]
display( df_sus.head() )

# Revisar transcrito unicas
print("Llaves unicas {}/{}".format( df_sus['Diagnostico_transcrito'].nunique(), df_sus.shape[0]))

# Revisar faltantes
print("Transcrito faltantes {}/{}".format( df_sus['Diagnostico_transcrito'].isna().sum(), df_sus.shape[0]))
print("Transcrito estandar {}/{}".format( df_sus['Diagnostico_estandar'].isna().sum(), df_sus.shape[0]))

# Diccionario
dic_sus = df_to_dict(df_sus)
#agregar llaves extendido
dic_sus.update( {d:d for d in df_ext['Diagnostico'].unique()} )
print("Diccionario diagnosticos", len( dic_sus ))

,Diagnostico_transcrito,Diagnostico_estandar,Diagnostico_extendido,Freq,Npartes,Nfalta,Sep1,Sep2,Sep3,Sep4,Sep5,Ext1,Ext2,Ext3,Ext4,Ext5,IDs
6,HPAFPC,HPAFC,herida arma fuego craneo,833,1,0,HPAFC,NaN,NaN,NaN,NaN,herida arma fuego craneo,NaN,NaN,NaN,NaN,"BO_1974_00037,BO_1974_00104,BO_1974_00162,BO_1..."
10,Quemaduras,Quem,quemaduras,697,1,0,Quem,NaN,NaN,NaN,NaN,quemaduras,NaN,NaN,NaN,NaN,"BO_1974_00034,BO_1974_00051,BO_1974_00089,BO_1..."
15,AA,AXA,asfixia por ahorcamiento,514,1,0,AXA,NaN,NaN,NaN,NaN,asfixia por ahorcamiento,NaN,NaN,NaN,NaN,"BO_1974_00025,BO_1974_00070,BO_1974_00100,BO_1..."
17,AQxT,AQXT,amputacion quirurgica por traumatismo,466,1,0,AQXT,NaN,NaN,NaN,NaN,amputacion quirurgica por traumatismo,NaN,NaN,NaN,NaN,"BO_1975_06705,BO_1975_06706,BO_1975_06710,BO_1..."
19,ANN,ANEN,asfixia neo natorum,404,1,0,ANEN,NaN,NaN,NaN,NaN,asfixia neo natorum,NaN,NaN,NaN,NaN,"BO_1974_02370,BO_1974_03518,BO_1974_04767,BO_1..."


Llaves unicas 2661/2661
Transcrito faltantes 0/2661
Transcrito estandar 0/2661
Diccionario diagnosticos 2905


### Regenerar tablas de frecuencia

Esto parte del diagnostico simplificado, aunque utiliza el separado para sustituciones complejas. Corre bajo tu propio riesgo

Separar y regenerar diagnostico transcrito

In [8]:
# Agrupar diagnostico y poner frecuencia y ejemplos
data = df.groupby('Diagnostico_transcrito').agg(
                      Freq=('ID', 'size'), 
                      IDs=('ID', first_n_to_text)
                 )
data = data.sort_values('Freq', ascending=False).reset_index()

# Remplazar complejas
data['Remplazo'] = data['Diagnostico_transcrito'].replace(dic_sus)
# Contar partes de cada diagnostico
data['Npartes'] = data['Remplazo'].str.count('\+') + 1

# Separar en partes
target_col = data['Remplazo']
sep = [break_and_simplify(s) for s in target_col.to_list()]
sep = pd.DataFrame(sep, index=target_col).fillna(nan)
sep_col = [f'Sep{i+1}' for i in range(sep.shape[1])]
sep.columns = sep_col
sep = sep.replace(dic_sus) # ! estandariza las partes usando sustitucion
sep

# Contar faltantes
sep['Nfalta'] = sep.apply(is_missing, dic=dic_sus).sum(axis=1)
sep['Diagnostico_estandar'] = sep[sep_col].apply(join_series, axis=1) # ! regenerar el estandar
sep = sep.reset_index()

# Unir 
data = pd.concat([data, sep], axis=1, join='inner').drop('Remplazo', axis=1)

# Agregar extendido
ext_col = list()
for col in sep_col:
    new_col = col.replace('Sep','Ext')
    data[new_col] = data[col].replace(dic_ext)
    ext_col.append(new_col)
data['Diagnostico_extendido'] = data[ext_col].apply(join_series, join_str=' ', axis=1)

# Ordenar
data = data[['Diagnostico_transcrito','Diagnostico_estandar','Diagnostico_extendido',
             'Freq','Npartes','Nfalta'] +sep_col +ext_col +['IDs']]

# Colorear faltantes
data_color = data.style.apply(highlight_missing, subset=sep_col, dic=dic_sus)
data_color.to_excel('catalogos/cat_diag_sustitucion_revisar.xlsx', index=False)
data_color
data['Nfalta'].sum()

<>:11: SyntaxWarning: invalid escape sequence '\+'
<>:11: SyntaxWarning: invalid escape sequence '\+'
/tmp/ipykernel_4925/4074908978.py:11: SyntaxWarning: invalid escape sequence '\+'
  data['Npartes'] = data['Remplazo'].str.count('\+') + 1


np.int64(1585)

In [9]:
data

,Diagnostico_transcrito,Diagnostico_estandar,Diagnostico_extendido,Freq,Npartes,Nfalta,Sep1,Sep2,Sep3,Sep4,Sep5,Ext1,Ext2,Ext3,Ext4,Ext5,IDs
0,S-D,S-D,sin datos,55878,1,0,S-D,NaN,NaN,NaN,NaN,sin datos,NaN,NaN,NaN,NaN,"BO_1968_00001,BO_1968_00002,BO_1968_00003,BO_1..."
1,TM,TM,traumatismo multiple,7505,1,0,TM,NaN,NaN,NaN,NaN,traumatismo multiple,NaN,NaN,NaN,NaN,"BO_1974_00019,BO_1974_00020,BO_1974_00026,BO_1..."
2,TCE,TCE,traumatismo craneo encefalico,5939,1,0,TCE,NaN,NaN,NaN,NaN,traumatismo craneo encefalico,NaN,NaN,NaN,NaN,"BO_1974_00018,BO_1974_00021,BO_1974_00030,BO_1..."
3,BN,BN,bronconeumonia,1826,1,0,BN,NaN,NaN,NaN,NaN,bronconeumonia,NaN,NaN,NaN,NaN,"BO_1974_00005,BO_1974_00008,BO_1974_00011,BO_1..."
4,CVG,CVG,congestion visceral generalizada,1287,1,0,CVG,NaN,NaN,NaN,NaN,congestion visceral generalizada,NaN,NaN,NaN,NaN,"BO_1974_00151,BO_1974_00165,BO_1974_00166,BO_1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3704,HPAF C/S Vasos,HPAF+AN,herida arma fuego anemia,1,2,0,HPAF,AN,NaN,NaN,NaN,herida arma fuego,anemia,NaN,NaN,NaN,BO_1975_01878
3705,HPAF C-Sec ArtF,HPAF+AN,herida arma fuego anemia,1,2,0,HPAF,AN,NaN,NaN,NaN,herida arma fuego,anemia,NaN,NaN,NaN,BO_1975_02943
3706,HPAF AQxT,AQXT,amputacion quirurgica por traumatismo,1,1,0,AQXT,NaN,NaN,NaN,NaN,amputacion quirurgica por traumatismo,NaN,NaN,NaN,NaN,BO_1979_07232
3707,HPAF (Cuello),HPAFCu,herida arma fuego cuello,1,1,0,HPAFCu,NaN,NaN,NaN,NaN,herida arma fuego cuello,NaN,NaN,NaN,NaN,BO_1974_00667


### Diagnosticos sencillos por frecuencia

In [10]:
sep = []
for col in sep_col :
    d = data[[col,'Freq']]
    d.columns = ['Sep', 'Freq']
    sep.append(d)
sep = pd.concat(sep, ignore_index=True)

sep = sep.dropna()
sep = sep.groupby('Sep').sum().reset_index() \
         .sort_values('Freq', ascending=False)
sep['Miss'] = is_missing(sep['Sep'], dic=dic_sus)

sep = sep.sort_values(['Freq','Sep'], ascending=[False, True])

sep.to_csv('catalogos/diag_partes_freq.csv', index=False)
sep

,Sep,Freq,Miss
908,S-D,55901,False
1028,TM,8020,False
972,TCE,6945,False
148,BN,5329,False
203,CH,1536,False
...,...,...,...
1357,septico,1,True
1358,shock,1,True
1359,trastorno,1,True
1360,trau,1,True


In [11]:
sep[sep['Miss']==True].head(25)

,Sep,Freq,Miss
768,P,59,True
320,E,36,True
236,CS,33,True
231,CQ,32,True
3,A,26,True
650,L,20,True
945,T,18,True
456,H,14,True
590,IP,14,True
669,M,14,True


## Revisar procedencias

### cat_proc_extendido.csv
* Procedencia | Extendido | Alcaldia | Frecuencia

In [12]:
df_ext = pd.read_csv('catalogos/cat_proc_extendido.csv')
df_ext

display(df_ext.tail())

# Revisar llaves unicas
print("Llaves unicas {}/{}".format( df_ext['Procedencia'].nunique(), df_ext.shape[0]))

# Revisar llaves unicas
print("Extend unicas {}/{}".format( df_ext['Extendido'].nunique(), df_ext.shape[0]))

# Faltantes
print('Faltantes:', df_ext.isna().sum())

dic_ext = df_to_dict(df_ext[['Procedencia','Extendido']])
len( dic_ext )

dic_cla = df_to_dict(df_ext[['Procedencia','Alcaldia']])
len( dic_cla )


,Procedencia,Extendido,Alcaldia
56,VC-2,Coord Territorial 2 Venustiano Carranza (Col M...,Venustiano Carranza
57,XO-1,Coord Territorial 1 Xochimilco - Reclusorio Su...,Xochimilco
58,XO-2,Coord Territorial 2 Xochimilco (Barrio de San ...,Xochimilco
59,PGR,Procuraduria General de la Republica,NaN
60,S-D,Sin datos,NaN


Llaves unicas 61/61
Extend unicas 61/61
Faltantes: Procedencia    0
Extendido      0
Alcaldia       2
dtype: int64


61

### cat_diag_sustitucion.xlsx
* Procedencia_transcrito | Procedencia_estandar | Freq | Npartes | Nfalta | Partes coloreadas | IDs

In [13]:
df_sus = pd.read_excel('catalogos/cat_proc_sustitucion.xlsx')
display( df_sus.head() )

# Revisar transcrito unicas
print("Llaves unicas {}/{}".format( df_sus['Procedencia_transcrito'].nunique(), df_sus.shape[0]))

# Revisar faltantes
print("Transcrito faltantes {}/{}".format( df_sus['Procedencia_transcrito'].isna().sum(), df_sus.shape[0]))
print("Transcrito estandar {}/{}".format( df_sus['Procedencia_estandar'].isna().sum(), df_sus.shape[0]))

# Diccionario
dic_sus = df_to_dict(df_sus)
#agregar llaves extendido
dic_sus.update( {d:d for d in df_ext['Procedencia'].unique()} )
print("Diccionario procedencias", len( dic_sus ))

,Procedencia_transcrito,Procedencia_estandar,Procedencia_extendido,Freq,Npartes,Nfalta,Sep1,Sep2,Sep3,Ext1,Ext2,Ext3,IDs
0,S-D,S-D,Sin datos,25779,1,0,S-D,NaN,NaN,Sin datos,NaN,NaN,"BO_1968_00001,BO_1968_00002,BO_1968_00003,BO_1..."
1,32a,COY-2,Coord Territorial 2 Coyoacan (Col Romero de Te...,2073,1,0,COY-2,NaN,NaN,Coord Territorial 2 Coyoacan (Col Romero de Te...,NaN,NaN,"BO_1977_00143,BO_1977_00175,BO_1977_00204,BO_1..."
2,33a,HB,Hospital Balbuena (Col Jardin Balbuena),1968,1,0,HB,NaN,NaN,Hospital Balbuena (Col Jardin Balbuena),NaN,NaN,"BO_1977_00177,BO_1977_00184,BO_1977_00185,BO_1..."
3,37a,CR,Cruz Roja (Col Polanco),1701,1,0,CR,NaN,NaN,Cruz Roja (Col Polanco),NaN,NaN,"BO_1977_00182,BO_1977_00196,BO_1977_00201,BO_1..."
4,32,COY-2,Coord Territorial 2 Coyoacan (Col Romero de Te...,1562,1,0,COY-2,NaN,NaN,Coord Territorial 2 Coyoacan (Col Romero de Te...,NaN,NaN,"BO_1977_05169,BO_1978_01205,BO_1980_00002,BO_1..."


Llaves unicas 859/859
Transcrito faltantes 0/859
Transcrito estandar 0/859
Diccionario procedencias 898


### Regenerar tablas de frecuencia

Esto parte del procedenciar simplificado, aunque utiliza el separado para sustituciones complejas. Corre bajo tu propio riesgo

Separar y regenerar procedencia transcrito

In [14]:
# Agrupar diagnostico y poner frecuencia y ejemplos
data = df.groupby('Procedencia_transcrito').agg(
                      Freq=('ID', 'size'), 
                      IDs=('ID', first_n_to_text)
                 )
data = data.sort_values('Freq', ascending=False).reset_index()

# Remplazar complejas
data['Remplazo'] = data['Procedencia_transcrito'].replace(dic_sus)
# Contar partes de cada procedencia
data['Npartes'] = data['Remplazo'].str.count('\+') + 1

# Separar en partes
target_col = data['Remplazo']
sep = [break_and_simplify(s) for s in target_col.to_list()]
sep = pd.DataFrame(sep, index=target_col).fillna(nan)
sep_col = [f'Sep{i+1}' for i in range(sep.shape[1])]
sep.columns = sep_col
sep = sep.replace(dic_sus) # ! estandariza las partes usando sustitucion
sep

# Contar faltantes
sep['Nfalta'] = sep.apply(is_missing, dic=dic_sus).sum(axis=1)
sep['Procedencia_estandar'] = sep[sep_col].apply(join_series, axis=1) # ! regenerar el estandar
sep = sep.reset_index()

# Unir 
data = pd.concat([data, sep], axis=1, join='inner').drop('Remplazo', axis=1)

# Agregar extendido
ext_col = list()
for col in sep_col:
    new_col = col.replace('Sep','Ext')
    data[new_col] = data[col].replace(dic_ext)
    ext_col.append(new_col)
data['Procedencia_extendido'] = data[ext_col].apply(join_series, join_str=' ', axis=1)

# Ordenar
data = data[['Procedencia_transcrito','Procedencia_estandar','Procedencia_extendido',
             'Freq','Npartes','Nfalta'] +sep_col +ext_col +['IDs']]

# Colorear faltantes
data_color = data.style.apply(highlight_missing, subset=sep_col, dic=dic_sus)
data_color.to_excel('catalogos/cat_proc_sustitucion_revisar.xlsx', index=False)
data_color
data['Nfalta'].sum()

<>:11: SyntaxWarning: invalid escape sequence '\+'
<>:11: SyntaxWarning: invalid escape sequence '\+'
/tmp/ipykernel_4925/2792538752.py:11: SyntaxWarning: invalid escape sequence '\+'
  data['Npartes'] = data['Remplazo'].str.count('\+') + 1


np.int64(24)

### Procedencias sencillos por frecuencia

In [15]:
sep = []
for col in sep_col :
    d = data[[col,'Freq']]
    d.columns = ['Sep', 'Freq']
    sep.append(d)
sep = pd.concat(sep, ignore_index=True)

sep = sep.dropna()
sep = sep.groupby('Sep').sum().reset_index() \
         .sort_values('Freq', ascending=False)
sep['Miss'] = is_missing(sep['Sep'], dic=dic_sus)

sep = sep.sort_values(['Freq','Sep'], ascending=[False, True])

sep.to_csv('catalogos/proc_partes_freq.csv', index=False)
sep

,Sep,Freq,Miss
126,S-D,25780,False
87,HB,5467,False
57,CR,4890,False
76,GAM-2,4112,False
55,COY-2,3638,False
...,...,...,...
129,T,1,False
130,T-1,1,False
131,TC-2,1,False
132,TEX,1,False


In [16]:
sep[sep['Miss']==True].head(25)

,Sep,Freq,Miss
138,VC,7,True
35,BJ,2,True
46,C1,2,True
53,COY,2,True
17,45a,1,True
43,C-CV,1,True
49,CD,1,True
50,CD-2,1,True
58,CRN,1,True
71,DP,1,True


Estandarizar a extendido

In [17]:
sep['Extendido'] = sep['Sep'].map(dic_ext)
sep

,Sep,Freq,Miss,Extendido
126,S-D,25780,False,Sin datos
87,HB,5467,False,Hospital Balbuena (Col Jardin Balbuena)
57,CR,4890,False,Cruz Roja (Col Polanco)
76,GAM-2,4112,False,Coord Territorial 2 Gustavo A Madero (Col Arag...
55,COY-2,3638,False,Coord Territorial 2 Coyoacan (Col Romero de Te...
...,...,...,...,...
129,T,1,False,NaN
130,T-1,1,False,NaN
131,TC-2,1,False,NaN
132,TEX,1,False,NaN


In [18]:
sep[sep['Extendido'].isna()].head(25)

,Sep,Freq,Miss,Extendido
127,SC,36,False,NaN
138,VC,7,True,NaN
109,MH,3,False,NaN
28,AH-VI,2,False,NaN
35,BJ,2,True,NaN
46,C1,2,True,NaN
53,COY,2,True,NaN
80,GAM-6,2,False,NaN
137,TLP-3,2,False,NaN
0,1136,1,False,NaN


### Diagnosticos astringente con alcaldia

In [19]:
data_filter = data[['Procedencia_transcrito'] + sep_col]
cla_col = [col.replace('Sep','Cla') for col in sep_col]

for sep, ext, cla in zip(sep_col, ext_col, cla_col):
    data[sep] = data[sep].map(dic_sus)
    data[ext] = data[sep].map(dic_ext)
    data[cla] = data[sep].map(dic_cla)

data['Procedencia_estandar'] = data[sep_col].apply(join_series, sort=True, axis=1)
data['Procedencia_extendido'] = data[ext_col].apply(join_series, sort=True, join_str=' + ', axis=1)
data['Procedencia_alcaldia'] = data[cla_col].apply(join_series, sort=True, join_str=' + ', axis=1)

data = data.drop(sep_col+ext_col+cla_col, axis=1)
data = data[data['Procedencia_estandar'].notna()]
data.to_csv('catalogos/cat_proc_sustitucion.csv', index=False)
data[ data['Nfalta']>0 ]


,Procedencia_transcrito,Procedencia_estandar,Procedencia_extendido,Freq,Npartes,Nfalta,IDs,Procedencia_alcaldia
450,Juarez 4a deleg,CUH-8,Coord Territorial 8 Cuauhtemoc (Col Obrera),1,2,1,BO_1973_02802,Cuauhtemoc
480,Hdz juarez 20a,IZP-5,Coord Territorial 5 Iztapalapa (Col V Guerrero),1,2,1,BO_1973_05439,Iztapalapa
494,IZA-VC-HB-,HB+IZP-1,Coord Territorial 1 Iztapalapa (Barrio de San ...,1,3,1,BO_1975_01723,Iztapalapa + Venustiano Carranza
497,HRL-VC/HB,HB+HGRL,Hospital Balbuena (Col Jardin Balbuena) + Hosp...,1,3,1,BO_1975_00092,Miguel Hidalgo + Venustiano Carranza
515,Tlalpan D. P.,TLP-1,Coord Territorial 1 Tlalpan (Col Ampliacion Mi...,1,2,1,BO_1971_02174,Tlalpan
532,PGR M-XIII,PGR,Procuraduria General de la Republica,1,2,1,BO_1977_04749,NaN
538,VC-CH,CUH-1,Coord Territorial 1 Cuauhtemoc (Col San Simon ...,1,2,1,BO_1977_00614,Cuauhtemoc
568,MH-VC-HB,HB,Hospital Balbuena (Col Jardin Balbuena),1,3,1,BO_1975_02956,Venustiano Carranza
595,MJ-CM,HCMS21,Hospital Centro Médico Nacional Siglo XXI (Col...,1,2,1,BO_1975_03941,Cuauhtemoc
621,C. R. N 2a Deleg,VC-1,Coord Territorial 1 Venustiano Carranza (Col M...,1,2,1,BO_1971_00924,Venustiano Carranza


Done